In [3]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from wh_research.data.hyperliquid.labels import add_forward_returns

# Choosing labels to predict

### I will choose the following labels as candidates

-  1-day forward log-return
-  3-day forward log-return
-  7-day forward log-return
-  1-day forward log-return direction
-  3-day forward log-return direction
-  7-day forward log-return direction

In [4]:
PROJECT_ROOT = Path.cwd().parent.parent
data_path = PROJECT_ROOT/"data/hyperliquid/case_study_data/universe_1d_min_mdv_2m_exit_1m_280323_171026.parquet"
df = pd.read_parquet(data_path)

### Computing the labels

`add_forward_returns` adds, for each horizon n, three columns:
- `fwd_log_return_{n}`: log(close n days later / this close), from this bar's `close_time`
- `fwd_direction_{n}`: 1 if that return is above 0, 0 if not, and missing where the return is missing
- `fwd_end_time_{n}`: when the label becomes known (used to purge the cross validation folds)

We compute them on every row first and only then keep the rows where `in_universe` is True, so the labels never skip over days when a symbol was outside the universe. The `fwd_` columns look into the future by design: they are never used as features.

In [5]:
# Add the forward log-return, direction and end-time columns for 1, 3 and 7 days.
# Done on the complete series of every symbol (all rows), before filtering to the universe.
df = add_forward_returns(df, "1d", [1, 3, 7])

#### Check the labels on the days in the universe

The last n days of every symbol have no label yet (the future has not happened), so they are missing. We do not drop them here.

In [6]:
labelled = df[df["in_universe"]]
horizons = [1, 3, 7]

print("Rows in the universe:", len(labelled))
print()
print("Missing labels:")
print(labelled[[f"fwd_log_return_{n}" for n in horizons]].isna().sum())
print()
print("Share of up days (missing labels skipped):")
print(labelled[[f"fwd_direction_{n}" for n in horizons]].mean())

Rows in the universe: 59645

Missing labels:
fwd_log_return_1     71
fwd_log_return_3    212
fwd_log_return_7    492
dtype: int64

Share of up days (missing labels skipped):
fwd_direction_1    0.469937
fwd_direction_3    0.459021
fwd_direction_7    0.437036
dtype: float64
